# Experimento completo (Colab, GPU T4)

Corre el experimento de punta a punta y deja las trazas en `resultados/`. El informe y la reproducción de tablas sin GPU están en `notebook.ipynb`.

| Paso | Qué hace | Tiempo aprox. en T4 |
|---|---|---|
| 0 | Preparar: GPU, repo, vLLM | 3–5 min |
| 1 | Prueba de formato (5 preguntas, 1.5B) | 3 min |
| 2 | Dev (20 preguntas, 1.5B y 7B) | 6 min |
| 3 | **Test** (160 preguntas, 1.5B, 3B y 7B): resultados del informe | 35–45 min |
| 4 | Extra: 14B (solo directa y react) | 30–35 min |
| 5 | Guardar resultados | 1 min |

Cada `correr.py` corre en un proceso aparte, porque vLLM no libera bien la GPU dentro del notebook. Cada corrida escribe `resultados/trazas_<modelo>_<split>.jsonl` y reemplaza solo las condiciones que vuelve a correr. Los pasos 1 y 2 son opcionales: los prompts se congelaron con dev, antes de correr test.

## 0. Preparar

Requiere GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clona el repo (o lo actualiza si ya estaba clonado en esta sesión).
import os
if not os.path.exists('/content/proyecto'):
    !git clone -q https://github.com/Felipedino/ReAct-Rag-Multimodal.git /content/proyecto
os.chdir('/content/proyecto')
!git pull -q
!git log --oneline -1

In [ ]:
# Versión probada: vllm 0.34.0 (trae torch 2.14.0+cu130).
!pip install -q vllm==0.34.0
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"

## 1. Prueba de formato (opcional)

5 preguntas escritas a mano (`data/prueba.jsonl`). Sirve para revisar a ojo el prompt y el parseo de las acciones. **No** mide desempeño.

Primero se arma el prompt de ReAct usando solo el tokenizer, sin cargar el modelo en la GPU:

In [ ]:
import json
from transformers import AutoTokenizer
import llm, agents

class SoloTokenizer:
    # Imita a vllm.LLM lo justo para armar prompts sin usar la GPU.
    def __init__(self, nombre):
        self.tok = AutoTokenizer.from_pretrained(nombre)
    def get_tokenizer(self):
        return self.tok

falso = SoloTokenizer(llm.MODELOS['1.5B'])
with open('data/prueba.jsonl', encoding='utf-8') as f:
    q = json.loads(f.readline())
usuario = f"{agents.INSTR_REACT}\nEjemplos:\n{agents.EJEMPLOS_REACT}\nPregunta: {q['pregunta']}"
p = llm.armar_prompt(falso, agents.SISTEMA, usuario, prefijo_asistente='Pensamiento 1:')
print(p)
print('Tokens del prompt ReAct (paso 1):', llm.contar_tokens(falso, p))

In [ ]:
!python correr.py 1.5B prueba
!python evaluate.py prueba

In [ ]:
# Trazas de la prueba: respuesta, cómo terminó y, en ReAct, la trayectoria completa.
with open('resultados/trazas_1.5B_prueba.jsonl', encoding='utf-8') as f:
    trazas = [json.loads(l) for l in f]
for t in trazas:
    print('=' * 100)
    print(f"[{t['condicion']}] {t['pregunta']}")
    print(f"gold: {t['respuesta_gold']} | modelo: {t['respuesta_modelo']!r} | "
          f"termino: {t['termino']} | acciones inválidas: {t['acciones_invalidas']}")
    if t['condicion'] == 'react':
        print(t['texto_modelo'])

Qué mirar:
- directa y rag: respuestas cortas, sin explicación.
- react: cada paso trae `Acción k: Buscar[...]` o `Terminar[...]`, y el modelo no inventa observaciones.
- `termino='contexto'` indica que hay que subir `llm.MAX_MODEL_LEN`.

## 2. Dev (opcional)

20 preguntas (5 por celda). Sirvió para revisar que todo funcione y congelar los prompts (DECISIONES 10).

In [ ]:
!python correr.py 1.5B dev
!python correr.py 7B dev
!python evaluate.py dev

In [ ]:
# ReAct del 1.5B que llegó al máximo de pasos (bucles): qué buscó en cada paso.
with open('resultados/trazas_1.5B_dev.jsonl', encoding='utf-8') as f:
    tr = [json.loads(l) for l in f]
bucles = [t for t in tr if t['condicion'] == 'react' and t['termino'] == 'max_pasos']
print(f'{len(bucles)} de {sum(t["condicion"] == "react" for t in tr)} trazas de ReAct llegan a 7 pasos')
for t in bucles[:3]:
    print('=' * 80); print(t['pregunta'], '| gold:', t['respuesta_gold'])
    for i, paso in enumerate(t['pasos'], 1):
        print(f"{i}. {paso['accion']}[{paso['argumento']}] -> {paso['titulo_observado']}")

## 3. Test (resultados del informe)

160 preguntas (40 por celda) y 3 condiciones (directa, rag, react). Un modelo por celda: si la sesión se corta, se retoma desde el modelo que falta. Al final de cada corrida, `correr.py` imprime el tiempo, los tokens generados y las llamadas reales a Wikipedia.

In [ ]:
!python correr.py 1.5B test     # ~10-12 min

In [ ]:
!python correr.py 3B test       # ~10 min

In [ ]:
!python correr.py 7B test       # ~11-13 min (incluye descarga)

In [ ]:
!python evaluate.py test

## 4. Extra: Qwen2.5-14B-Instruct-AWQ (opcional)

No es parte del diseño original (1.5B, 3B y 7B); sirve para ver si la tendencia con el tamaño sigue. Tiene tres diferencias con el resto:

- **Solo directa y react.** RAG fijo se omite por tiempo; ReAct solo toma ~26 min.
- **`MAX_MODEL_LEN = 6144`** en vez de 8192. El 14B pesa ~9,6 GB y deja ~7.000 tokens de caché KV (con `gpu_memory_utilization=0.95`, ya fijado en `llm.py`). vLLM exige que quepa al menos una petición del largo máximo. Consecuencia: ReAct puede cortarse por contexto antes que en los otros modelos (ver `termino == 'contexto'` en las trazas).
- El cambio de largo se hace con `sed` sobre `llm.py` solo para esta corrida, y después se restaura con `git checkout`.

Tiempo: ~4 min de carga + ~26 min de generación.

In [ ]:
!sed -i 's/^MAX_MODEL_LEN = 8192/MAX_MODEL_LEN = 6144/' llm.py
!grep -n "^MAX_MODEL_LEN" llm.py
!python correr.py 14B test directa react
!git checkout llm.py              # vuelve a 8192 para el resto
!python evaluate.py test

In [ ]:
# ¿Cuántas trazas de ReAct del 14B se cortaron por contexto?
import collections
with open('resultados/trazas_14B_test.jsonl', encoding='utf-8') as f:
    t14 = [json.loads(l) for l in f]
print(collections.Counter(t['termino'] for t in t14 if t['condicion'] == 'react'))

## 5. Guardar resultados

Colab borra todo al cerrar la sesión. Descarga `resultados/` (trazas, tablas y figuras) y `cache/` (las páginas nuevas de Wikipedia que se consultaron). Después se copian al repo local y se suben con git.

In [ ]:
!zip -qr salida.zip resultados cache
from google.colab import files
files.download('salida.zip')